In [ ]:
from sqlalchemy import create_engine

db_params = {
    'host': 'localhost',
    'database': 'postgres',
    'user': 'postgres',
    'password': 'postgres',
    'port': 5432
}

conn = create_engine(f"postgresql://{db_params['user']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['database']}")

## Runtime requirements for test generalization per project

In [ ]:
import pandas as pd

# Query to get test runtimes (not generalizations)
query = """
SELECT 
    p.id as project_id,
    p.root_path,
    j.stage,
    j.runtime
FROM junit_test_report j
JOIN project p ON j.project_id = p.id
WHERE j.variant IS NULL AND p.runtime IS NOT NULL  -- Original tests only, not generalizations
"""

# Load data into a DataFrame
df_all_runtimes = pd.read_sql_query(query, conn)

# Extract project name from root_path for better readability
df_all_runtimes['project_name'] = df_all_runtimes['root_path'].apply(lambda x: x.split('/')[-1])

# Group by project and stage, then calculate statistics
test_runtime_stats = df_all_runtimes.groupby(['project_id', 'project_name', 'stage']).agg(
    test_count=('runtime', 'count'),
    total_runtime=('runtime', 'sum'),
    avg_runtime=('runtime', 'mean'),
    median_runtime=('runtime', 'median'),
    min_runtime=('runtime', 'min'),
    max_runtime=('runtime', 'max'),
).reset_index()

# Round numeric columns for better readability
numeric_cols = ['avg_runtime', 'median_runtime', 'min_runtime', 'max_runtime', 'total_runtime']
for col in numeric_cols:
    test_runtime_stats[col] = test_runtime_stats[col].round(3)

# Sort by project name and stage
test_runtime_stats = test_runtime_stats.sort_values(['project_name', 'project_id', 'stage'])

# Display the result
print("Test Runtime Statistics by Project and Stage:")
display(test_runtime_stats)


In [ ]:
import pandas as pd

# Query to get generalization runtimes
query = """
SELECT 
    p.id as project_id,
    p.root_path,
    j.stage,
    j.variant,
    j.runtime
FROM junit_test_report j
JOIN project p ON j.project_id = p.id
WHERE j.variant IS NOT NULL AND p.runtime IS NOT NULL  -- Only generalizations (with variant)
"""

# Load data into a DataFrame
df_gen_runtimes = pd.read_sql_query(query, conn)

# Extract project name from root_path for better readability
df_gen_runtimes['project_name'] = df_gen_runtimes['root_path'].apply(lambda x: x.split('/')[-1])

# Group by project, stage, and variant, then calculate statistics
gen_runtime_stats = df_gen_runtimes.groupby(['project_id', 'project_name', 'stage', 'variant']).agg(
    test_count=('runtime', 'count'),
    total_runtime=('runtime', 'sum'),
    avg_runtime=('runtime', 'mean'),
    median_runtime=('runtime', 'median'),
    min_runtime=('runtime', 'min'),
    max_runtime=('runtime', 'max'),
).reset_index()

# Round numeric columns for better readability
numeric_cols = ['avg_runtime', 'median_runtime', 'min_runtime', 'max_runtime', 'total_runtime']
for col in numeric_cols:
    gen_runtime_stats[col] = gen_runtime_stats[col].round(3)

# Sort by project_id, stage, and variant
gen_runtime_stats = gen_runtime_stats.sort_values(['project_name', 'project_id', 'stage', 'variant'])

# Display the result
print("Generalization Runtime Statistics by Project, Stage, and Variant:")
display(gen_runtime_stats)


In [ ]:
import pandas as pd
import numpy as np

# Filter test data to only include the INITIAL stage as baseline
baseline_tests = test_runtime_stats[test_runtime_stats['stage'] == 'COLLECT_JUNIT_REPORTS_INITIAL'].copy()
baseline_tests['join_key'] = baseline_tests['project_id'].astype(str)  # Just project_id for joining

# Prepare the generalization data with a key for joining
gen_for_join = gen_runtime_stats.copy()
gen_for_join['join_key'] = gen_for_join['project_id'].astype(str)  # Just project_id for joining

# Create a list to store comparison results
comparison_rows = []

# For each generalization variant
for _, gen_row in gen_for_join.iterrows():
    # Find matching baseline test data
    matching_baseline = baseline_tests[baseline_tests['join_key'] == gen_row['join_key']]

    if not matching_baseline.empty:
        baseline_row = matching_baseline.iloc[0]

        # Calculate speedup ratios (gen/baseline)
        total_runtime_ratio = gen_row['total_runtime'] / baseline_row['total_runtime'] if baseline_row['total_runtime'] > 0 else np.nan
        avg_runtime_ratio = gen_row['avg_runtime'] / baseline_row['avg_runtime'] if baseline_row['avg_runtime'] > 0 else np.nan
        median_runtime_ratio = gen_row['median_runtime'] / baseline_row['median_runtime'] if baseline_row['median_runtime'] > 0 else np.nan

        # Add to comparison results
        comparison_rows.append({
            'project_id': gen_row['project_id'],
            'project_name': gen_row['project_name'],
            'stage': gen_row['stage'],
            'variant': gen_row['variant'],
            'test_count_ratio': gen_row['test_count'] / baseline_row['test_count'],
            'total_runtime_ratio': total_runtime_ratio,
            'avg_runtime_ratio': avg_runtime_ratio,
            'median_runtime_ratio': median_runtime_ratio,
        })

# Create the comparison dataframe
speedup_df = pd.DataFrame(comparison_rows)

# Round numeric columns for better readability
ratio_cols = [col for col in speedup_df.columns if 'ratio' in col]
diff_cols = [col for col in speedup_df.columns if 'diff' in col]

for col in ratio_cols:
    speedup_df[col] = speedup_df[col].round(3)

for col in diff_cols:
    speedup_df[col] = speedup_df[col].round(3)

# Sort by project name, stage, and variant
speedup_df = speedup_df.sort_values(['project_name', 'project_id', 'stage', 'variant'])

# Display the result
print("Speedup/Slowdown Comparison (Generalization vs. INITIAL Tests):")
print("Ratio < 1: Generalization is faster, Ratio > 1: Generalization is slower")
display(speedup_df)
